# 🔬 DermaFusion-AI — Kaggle Training Notebook
## EVA-02 Large + ConvNeXt V2 | Dual-Branch Fusion | SOTA 2026
### ⚙️ Before Running:
1. Datasets already added ✅
2. **Accelerator → GPU T4 x2** ✅
3. **Persistence → Files only** ✅
4. Run cells top to bottom — Steps 5+6 are combined to auto-start classifier after segmentation

## Step 0: Inspect Input Paths

In [ ]:
import os

def explore(path, depth=0, max_depth=2):
    if depth > max_depth or not os.path.exists(path):
        return
    for item in sorted(os.listdir(path)):
        full = os.path.join(path, item)
        kind = '📁' if os.path.isdir(full) else '📄'
        print('  ' * depth + f'{kind} {item}')
        if os.path.isdir(full):
            explore(full, depth + 1, max_depth)

explore('/kaggle/input', max_depth=2)

## Step 1: Clone / Pull Repo

In [1]:
import os
if not os.path.exists('/kaggle/working/DermaFusion-AI'):
    !git clone https://github.com/ai-with-abdullah/DermaFusion-AI.git /kaggle/working/DermaFusion-AI
else:
    !cd /kaggle/working/DermaFusion-AI && git fetch origin && git reset --hard origin/main

%cd /kaggle/working/DermaFusion-AI
print('Working directory:', os.getcwd())


Cloning into '/kaggle/working/DermaFusion-AI'...
remote: Enumerating objects: 2638, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 2638 (delta 12), reused 14 (delta 6), pack-reused 2615 (from 2)
Receiving objects: 100% (2638/2638), 186.86 MiB | 19.84 MiB/s, done.
Resolving deltas: 100% (285/285), done.
/kaggle/working/DermaFusion-AI
Working directory: /kaggle/working/DermaFusion-AI


## Step 2: Install Dependencies

In [2]:
%pip install -q timm>=0.9.12 albumentations>=1.3.1 opencv-python-headless scikit-learn scipy tqdm
print('✅ Dependencies installed')

Note: you may need to restart the kernel to use updated packages.
✅ Dependencies installed


## Step 3: Verify GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'✅ GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')
else:
    print('❌ No GPU — Settings > Accelerator > GPU T4 x2')

## Step 4.5: 🔥 Smoke Test (RUN BEFORE TRAINING)
Builds the real EVA-02 / ConvNeXt / Swin-UNet backbones (pretrained=False, no download), runs 1 segmentation step + 1 classifier step with the new spatial fusion + SALA, and checks shapes / gradients / NaNs. ~1-2 min. If this fails, DO NOT start full training — a shape bug would waste GPU quota.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python scripts/kaggle_smoke_test.py

## Step 4: Link All Datasets → data/ folder

In [1]:
import os

data_dir = '/kaggle/working/DermaFusion-AI/data'
os.makedirs(data_dir, exist_ok=True)

search_roots = ['/kaggle/input/competitions']
datasets_dir = '/kaggle/input/datasets'
if os.path.exists(datasets_dir):
    for username in os.listdir(datasets_dir):
        user_path = os.path.join(datasets_dir, username)
        if os.path.isdir(user_path):
            search_roots.append(user_path)
search_roots.append('/kaggle/input')

def find_folder(keywords, avoid_keywords=None):
    # Search up to 3 levels deep in /kaggle/input
    paths_to_check = ['/kaggle/input']
    checked_paths = set()
    for depth in range(3):
        next_paths = []
        for p in paths_to_check:
            if not os.path.exists(p) or p in checked_paths:
                continue
            checked_paths.add(p)
            try:
                for item in os.listdir(p):
                    full = os.path.join(p, item)
                    if os.path.isdir(full):
                        if any(kw in item.lower() for kw in keywords):
                            if avoid_keywords and any(akw in item.lower() for akw in avoid_keywords):
                                continue
                            # Check for a deeper subfolder that matches
                            try:
                                for sub in os.listdir(full):
                                    sub_full = os.path.join(full, sub)
                                    if os.path.isdir(sub_full) and any(kw in sub.lower() for kw in keywords):
                                        if avoid_keywords and any(akw in sub.lower() for akw in avoid_keywords):
                                            continue
                                        return sub_full
                            except Exception:
                                pass
                            return full
                        next_paths.append(full)
            except Exception:
                pass
        paths_to_check = next_paths
    return None
def link(src, dst_name, label):
    dst = os.path.join(data_dir, dst_name)
    if os.path.lexists(dst):
        try:
            if os.path.islink(dst):
                os.unlink(dst)
            elif os.path.isdir(dst):
                import shutil
                shutil.rmtree(dst)
            else:
                os.remove(dst)
        except Exception:
            pass
    if src:
        os.symlink(src, dst)
        print(f'✅ {label}: linked from {src}')
    else:
        print(f'❌ {label}: NOT FOUND')
link(find_folder(['skin-cancer-mnist', 'mnist-ham10000'], avoid_keywords=['part_', 'part1', 'part2', 'images', 'masks', 'segmentation', 'lesion-segmentations']),               'ham10000',  'HAM10000')
link(find_folder(['masks', 'segmentation', 'lesion-segmentations']), 'masks', 'HAM10000 Masks')
link(find_folder(['isic-2019', 'isic2019', 'skin-lesion-images', 'andrewmvd', 'salviohexia']), 'isic_2019', 'ISIC 2019')
link(find_folder(['siim-isic', 'melanoma-classification']),        'isic_2020', 'ISIC 2020')
link(find_folder(['isic-2024', 'skin-cancer-detection-3d']),       'isic_2024', 'ISIC 2024')
link(find_folder(['pad-ufes', 'padufes']),                         'pad_ufes',  'PAD-UFES-20')
link(find_folder(['ddi', 'diverse-dermatology']),                  'ddi',       'Stanford DDI')

derm7pt_src = find_folder(['derm7pt', 'release-v0', 'release_v0'])
if derm7pt_src:
    dst = '/kaggle/working/DermaFusion-AI/release_v0'
    if not os.path.exists(dst):
        os.symlink(derm7pt_src, dst)
        print(f'✅ DERM7PT: linked to {dst}')

print('\n📁 data/ contents:', os.listdir(data_dir))

✅ HAM10000: linked from /kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000
✅ HAM10000 Masks: linked from /kaggle/input/datasets/surajghuwalewala/ham1000-segmentation-and-classification/masks
✅ ISIC 2019: linked from /kaggle/input/datasets/andrewmvd/isic-2019
✅ ISIC 2020: linked from /kaggle/input/competitions/siim-isic-melanoma-classification
✅ ISIC 2024: linked from /kaggle/input/competitions/isic-2024-challenge
❌ PAD-UFES-20: NOT FOUND
✅ Stanford DDI: linked from /kaggle/input/datasets/ahmadali531/datasetddi

📁 data/ contents: ['images', 'isic_2024', 'isic_2020', 'ham10000', 'masks', 'isic_2019', 'ddi']


# Move the Weights and chack pints form the Dataset to Weights folder in the OutPut folder

In [2]:
import shutil, os

# Your input dataset path
# (check exact name you gave it)
!ls /kaggle/input/

# Then restore weights
src = '/kaggle/input/datasets/ahmadali531/weights'  # change if different name
dst = '/kaggle/working/DermaFusion-AI/outputs/weights'
os.makedirs(dst, exist_ok=True)

for f in os.listdir(src):
    shutil.copy2(f'{src}/{f}', f'{dst}/{f}')
    print(f'✅ Restored: {f}')

print("\n📁 Weights folder now contains:")
for f in os.listdir(dst):
    size = os.path.getsize(f'{dst}/{f}') / (1024*1024)
    print(f"   {f}: {size:.1f} MB")

competitions  datasets
✅ Restored: best_unet.pth

📁 Weights folder now contains:
   best_unet.pth: 364.4 MB


## Step 5: Phase 1 — Segmentation Training (Swin-UNet)
Trains the Swin-Transformer U-Net to perform lesion masking. Saves checkpoints after each epoch to `/kaggle/working/DermaFusion-AI/outputs/weights/resume_seg_checkpoint.pth`. If the session stops, re-running this cell will automatically resume from the last epoch.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
print('=' * 60)
print('🚀 PHASE 1 — Segmentation Training (25 epochs)')
print('   Swin-Tiny UNet | SEG_BATCH_SIZE=8 | 2× T4 GPUs')
print('=' * 60)
!PYTHONPATH=. python train_segmentation.py 2>&1 | tee /kaggle/working/train_segmentation.log
print('\n✅ Phase 1 complete!')

## Step 6: Phase 2 — Classifier Training & Ablations
To run the Ablation Study by training each model from scratch, you can execute the cells below one-by-one. Each cell trains a specific configuration and saves its weights separately so they do not overwrite each other.

### Step 6a: Ablation 1 — No TTA Training (Uses standard training but saves separately)
Estimated training time: ~2 hours per epoch. Trains the full model structure but saves to `best_classifier_no_tta.pth`.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python train_classifier.py --ablation no_tta 2>&1 | tee /kaggle/working/train_no_tta.log
print('\n✅ Ablation 1 training complete!')

### Step 6b: Ablation 2 — ConvNeXt Only Training (Bypasses EVA-02 and fusion)
Estimated training time: ~30 minutes per epoch (much faster as it is a single backbone!). Saves to `best_classifier_convnext_only.pth`.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python train_classifier.py --ablation convnext_only 2>&1 | tee /kaggle/working/train_convnext_only.log
print('\n✅ Ablation 2 training complete!')

### Step 6c: Ablation 3 — EVA-02 Only Training (Bypasses ConvNeXt and fusion)
Estimated training time: ~1.2 hours per epoch. Saves to `best_classifier_eva_only.pth`.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python train_classifier.py --ablation eva_only 2>&1 | tee /kaggle/working/train_eva_only.log
print('\n✅ Ablation 3 training complete!')

### Step 6d: Ablation 4 — No Cross-Attention Training (Fuses via simple average)
Estimated training time: ~2 hours per epoch. Saves to `best_classifier_no_attention.pth`.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python train_classifier.py --ablation no_attention 2>&1 | tee /kaggle/working/train_no_attention.log
print('\n✅ Ablation 4 training complete!')

### Step 6e: Ablation 5 — No Segmentation Training (Original image to ConvNeXt)
Estimated training time: ~2 hours per epoch. Saves to `best_classifier_no_segmentation.pth`.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python train_classifier.py --ablation no_segmentation 2>&1 | tee /kaggle/working/train_no_segmentation.log
print('\n✅ Ablation 5 training complete!')

### Step 6f: Full Model Training (Dual-Branch + Segmentation + Cross-Attention + TTA)
Estimated training time: ~2 hours per epoch. This is your main final model. Saves to `best_dual_branch_fusion.pth`.

In [2]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python train_classifier.py 2>&1 | tee /kaggle/working/train_classifier.log
print('\n✅ Full Model training complete!')

/kaggle/working/DermaFusion-AI
Global seed set to: 42
2026-06-18 03:34:51,374 INFO ======================================================================
2026-06-18 03:34:51,374 INFO Starting Dual-Branch Fusion Classifier Training — 2026 SOTA Upgrade
2026-06-18 03:34:51,374 INFO   Branch A: EVA-02        → eva02_large_patch14_448.mim_in22k_ft_in22k_in1k
2026-06-18 03:34:51,374 INFO   Branch B: ConvNeXt V2   → convnextv2_base.fcmae_ft_in22k_in1k_384
2026-06-18 03:34:51,375 INFO   Multi-dataset:          True
2026-06-18 03:34:51,375 INFO   CutMix enabled:         True
2026-06-18 03:34:51,375 INFO   LR warmup epochs:       3
2026-06-18 03:34:51,375 INFO   Label smoothing:        0.1
2026-06-18 03:34:51,375 INFO   Device: cuda
2026-06-18 03:34:51,375 INFO ======================================================================
/kaggle/working/DermaFusion-AI/datasets/unified_dataset.py:458: DtypeWarning: Columns (51,52) have mixed types. Specify dtype option on import or set low_memory=False.

# Push the weights to the Hugging face

In [4]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
import os

token = UserSecretsClient().get_secret("HF_TOKEN")     # must match your secret name
REPO  = "ai-with-abdullah/dermafusion-ckpt"            # <-- put YOUR Hugging Face username
api = HfApi(token=token)
api.create_repo(repo_id=REPO, repo_type="model", private=True, exist_ok=True)

W = "/kaggle/working/DermaFusion-AI/outputs/weights"
for f in ["resume_checkpoint.pth", "best_unet.pth", "best_dual_branch_fusion.pth"]:
    if os.path.exists(f"{W}/{f}"):
        print("uploading", f, "...")
        api.upload_file(path_or_fileobj=f"{W}/{f}", path_in_repo=f, repo_id=REPO, repo_type="model")
        print("✅", f)
print("Done — checkpoints safe on Hugging Face. Quota can end now, nothing is lost.")

uploading resume_checkpoint.pth ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ resume_checkpoint.pth
uploading best_unet.pth ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ best_unet.pth
uploading best_dual_branch_fusion.pth ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ best_dual_branch_fusion.pth
Done — checkpoints safe on Hugging Face. Quota can end now, nothing is lost.


# Restore the Model files from the hagging face

In [ ]:
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
import shutil, os
token = UserSecretsClient().get_secret("HF_TOKEN")
REPO = "YOUR_HF_USERNAME/dermafusion-ckpt"
W = "/kaggle/working/DermaFusion-AI/outputs/weights"; os.makedirs(W, exist_ok=True)
for f in ["resume_checkpoint.pth", "best_unet.pth", "best_dual_branch_fusion.pth"]:
    p = hf_hub_download(repo_id=REPO, filename=f, token=token)
    shutil.copy(p, W); print("restored", f)

### Step 6g: SALA Ablation — train WITHOUT Source-Aware Logit Adjustment (Novelty #1 ablation)
Trains a second full model with `--no-sala`. Saves to `best_dual_branch_fusion_nosala.pth` so the main model is NOT overwritten. Compare its metrics vs the full model to quantify SALA's gain.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python train_classifier.py --no-sala 2>&1 | tee /kaggle/working/train_nosala.log
print('\n✅ SALA-ablation (no-SALA) training complete!')

## Step 7: Export Weights, Results and Plots to Output Tab
Copies the trained model weights, the **Ablation Study results CSV**, the **Evaluation Dashboard**, and all individual plots to `/kaggle/working/` so they appear in the Kaggle Output panel on the right for easy download.

In [ ]:
import shutil, os

# 1. Export Weights
weights_src = '/kaggle/working/DermaFusion-AI/outputs/weights/'
if os.path.exists(weights_src):
    for f in os.listdir(weights_src):
        if f.endswith('.pth'):
            dst = f'/kaggle/working/{f}'
            shutil.copy(os.path.join(weights_src, f), dst)
            print(f'✅ Weights: {f}  ({os.path.getsize(dst)/1e6:.0f} MB)')
else:
    print('⚠️ No weights folder found (yet)')

# 2. Export Ablation CSV Results
csv_path = '/kaggle/working/DermaFusion-AI/outputs/ablation_study_results.csv'
if os.path.exists(csv_path):
    shutil.copy(csv_path, '/kaggle/working/ablation_study_results.csv')
    print('✅ Results: ablation_study_results.csv')

# 3. Export Plots and Dashboard
plots_src = '/kaggle/working/DermaFusion-AI/outputs/plots/'
if os.path.exists(plots_src):
    for f in os.listdir(plots_src):
        if f.endswith(('.png', '.jpg', '.jpeg')):
            shutil.copy(os.path.join(plots_src, f), f'/kaggle/working/{f}')
            print(f'✅ Plot: {f}')

print('\n📦 Files exported to /kaggle/working/ and ready for download!')

## Step 8: Evaluation & Ablation Study Results
You can evaluate each of your trained ablation configurations individually using the cells below. Make sure you have completed the corresponding training step first so that the weights exist.

### Step 8a: Evaluate Ablation 1 — No TTA (Evaluates the standard weights without TTA)

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python evaluate.py --ablation no_tta 2>&1 | tee /kaggle/working/evaluate_no_tta.log
print('\n✅ Ablation 1 evaluation complete!')

### Step 8b: Evaluate Ablation 2 — ConvNeXt Only (Bypasses EVA-02)

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python evaluate.py --ablation convnext_only 2>&1 | tee /kaggle/working/evaluate_convnext_only.log
print('\n✅ Ablation 2 evaluation complete!')

### Step 8c: Evaluate Ablation 3 — EVA-02 Only (Bypasses ConvNeXt)

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python evaluate.py --ablation eva_only 2>&1 | tee /kaggle/working/evaluate_eva_only.log
print('\n✅ Ablation 3 evaluation complete!')

### Step 8d: Evaluate Ablation 4 — No Cross-Attention (Fuses via simple average)

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python evaluate.py --ablation no_attention 2>&1 | tee /kaggle/working/evaluate_no_attention.log
print('\n✅ Ablation 4 evaluation complete!')

### Step 8e: Evaluate Ablation 5 — No Segmentation (Original image passed to ConvNeXt)

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python evaluate.py --ablation no_segmentation 2>&1 | tee /kaggle/working/evaluate_no_segmentation.log
print('\n✅ Ablation 5 evaluation complete!')

### Step 8f: Evaluate Full Model (Main Dual-Branch + TTA)
Running this cell evaluates the final model, runs the Ablation Study comparison script, and generates the final dashboard.

In [5]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python evaluate.py 2>&1 | tee /kaggle/working/evaluate_full.log
print('\n✅ Full Model evaluation and summary dashboard complete!')

/kaggle/working/DermaFusion-AI
Global seed set to: 42
2026-06-18 10:03:32,771 INFO Starting Dual-Branch Fusion Evaluation (TTA enabled)
/kaggle/working/DermaFusion-AI/datasets/unified_dataset.py:458: DtypeWarning: Columns (51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)

[UnifiedDataset] Scanning available datasets...
  [HAM10000] Loaded 10015 records (fmt=standard).
  [DEBUG ISIC 2019] base: /kaggle/working/DermaFusion-AI/data/isic_2019
  [DEBUG ISIC 2019] base contents: ['ISIC_2019_Training_GroundTruth.csv', 'ISIC_2019_Training_Metadata.csv', 'ISIC_2019_Training_Input']
  [DEBUG ISIC 2019] subdirectory ISIC_2019_Training_Input contents: ['ISIC_2019_Training_Input']
  [DEBUG ISIC 2019] subdirectory ISIC_2019_Training_Input contents: ['ISIC_0057312.jpg', 'ISIC_0014233_downsampled.jpg', 'ISIC_0059626.jpg', 'ISIC_0056156.jpg', 'ISIC_0030912.jpg']
  [DEBUG ISIC 2019] all_search_dirs: ['/kaggle/working/DermaFusion-AI/data/isic_2

### Step 8g: Evaluate SALA Ablation (no-SALA model)
Evaluates `best_dual_branch_fusion_nosala.pth`. Novelty #1 contribution = (Full Model metrics − this). The architecture ablations (No-Uncertainty-Bias, No-Mirror-Asymmetry, Plain-Spatial) run automatically inside the Full-Model eval above via run_ablation_study.py → `outputs/ablation_study_results.csv`.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python evaluate.py --no-sala 2>&1 | tee /kaggle/working/evaluate_nosala.log
print('\n✅ SALA-ablation evaluation complete!')

## Step 9: Advanced Paper Evaluations
Runs calibration, bootstrap confidence intervals, McNemar significance tests, inference benchmarking, 5-fold CV, and external smartphone/dermoscopy evaluations (PAD-UFES-20 and DERM7PT).

In [ ]:
%cd /kaggle/working/DermaFusion-AI

print('=== 1. Probability Calibration (Temperature Scaling) ===')
!PYTHONPATH=. python -m evaluation.run_temperature_scaling

print('\n=== 2. Statistical Significance (McNemar\'s Test) ===')
!PYTHONPATH=. python -m evaluation.run_statistical_tests

print('\n=== 3. Statistical Confidence Intervals (Bootstrap) ===')
!PYTHONPATH=. python -m evaluation.run_confidence_intervals

print('\n=== 4. Inference Latency & Speed Benchmark ===')
!PYTHONPATH=. python -m evaluation.run_inference_benchmark

print('\n=== 5. 5-Fold Cross-Validation (EVA-02 Small) ===')
!PYTHONPATH=. python -m evaluation.run_cross_validation

print('\n=== 6. Smartphone External Test (PAD-UFES-20) ===')
!PYTHONPATH=. python test_padufes.py

print('\n=== 7. Smartphone Head Fine-Tuning ===')
!PYTHONPATH=. python finetune_padufes.py

print('\n=== 8. DERM7PT External Test (4 combinations) ===')
!PYTHONPATH=. python test_both_weights.py

print('\n✅ All advanced paper evaluations are complete!')